<a href="https://colab.research.google.com/github/aiwithpavansama/LLM-Model-Creation-from-Scratch/blob/main/LLm_From_Scratch_PreTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# stage 1 : data preparation and sampling , attention mechanism , LLM architecture , Building an LLM

  # 25 concepts to learn in AI
  https://thecuriousmak.substack.com/p/25-ai-concepts-people-use-but-dont

In [2]:
#tokanization cncept
# step 1: splitting the sentences into individual tokens
# step 2: given the tokens to ID's
# step 3: Convert this into embiddings

In [3]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 6.9 MB/s eta 0:00:00


In [4]:
from pypdf import PdfReader

In [5]:
# reading the text from the PDF and then getting first 100 characters
reader = PdfReader("/content/drive/MyDrive/The_Verdict.pdf")   # or "/path/to/file.pdf"

text = ""
for page in reader.pages:
    text += page.extract_text() or ""

print("total number of characters",len(text))
print(text[:99])

total number of characters 21813
1
The Verdict
Edith Wharton
1908
Exported from Wikisource on May 20, 20242
I HAD always thought Jac


In [6]:
#splitting the words based on special characters and removing the spaces.
import re
result=re.split(r'([,.:;?()\'"]|--|\s)',text)
result = [item for item in result if item.strip()]
print(len(result))
print(result[:99])

4871
['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20', ',', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called']


In [7]:
preprocessed=re.split(r'([,.:;?()\'"]|--|\s)',text)
preprocessed=[item for item in preprocessed if item.strip()]
print(len(preprocessed))
print(result[:30])

4871
['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20', ',', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough']


In [8]:
# step 2: given the tokens to ID's (first converting from words to numbers)
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print("total number of unique words",vocab_size)


total number of unique words 1276


In [9]:
vocab={token:Integer for Integer,token in enumerate(all_words)}

In [10]:
for i, item in enumerate(vocab.items()):
  print(item)
  if i>=50:
    break

('"', 0)
("'", 1)
('(', 2)
(')', 3)
(',', 4)
('--', 5)
('.', 6)
('0', 7)
('1', 8)
('10', 9)
('14', 10)
('19', 11)
('1908', 12)
('1929', 13)
('20', 14)
('2021', 15)
('20242', 16)
('4', 17)
('5', 18)
(':', 19)
(';', 20)
('?', 21)
('A', 22)
('Abigor', 23)
('About', 24)
('AdamBMorgan', 25)
('Ah', 26)
('Among', 27)
('And', 28)
('Are', 29)
('Arrt', 30)
('As', 31)
('At', 32)
('Attribution-ShareAlike', 33)
('AzaToth', 34)
('Be', 35)
('Begin', 36)
('Bender235', 37)
('Blurpeace', 38)
('Boris23', 39)
('Bromskloss', 40)
('Burlington', 41)
('But', 42)
('By', 43)
('Carlo', 44)
('Chicago', 45)
('Claude', 46)
('Come', 47)
('Commons', 48)
('Creative', 49)
('Croft', 50)


In [11]:
class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is not None:
            self.str_to_int = vocab
            self.int_to_str = {idx: token for token, idx in vocab.items()}
        else:
            self.str_to_int = {}
            self.int_to_str = {}

    def encode(self, tokens):
        # Convert text tokens to token IDs
        encoded_ids = []
        for token in tokens:
            if token in self.str_to_int:
                encoded_ids.append(self.str_to_int[token])
            elif '<|unk|>' in self.str_to_int:
                encoded_ids.append(self.str_to_int['<|unk|>'])
        return encoded_ids

    def decode(self, token_ids):
        # Convert token IDs back to tokens
        tokens = [self.int_to_str[idx] for idx in token_ids if idx in self.int_to_str]
        text = " ".join(tokens)
        # Clean up spaces before punctuation marks
        import re
        text = re.sub(r"\s+([,.:;?(!'\"])", r"\1", text)
        return text

# Instantiate and test the tokenizer with the updated vocabulary
tokenizer = SimpleTokenizer(vocab=vocab)

# Example usage with both known and unknown words
test_tokens = preprocessed[:10] + ["<|endoftext|>", "UnknownWordExample"]
print("Original Tokens:", test_tokens)

token_ids = tokenizer.encode(test_tokens)
print("Encoded IDs:", token_ids)

decoded_text = tokenizer.decode(token_ids)
print("Decoded Text:", decoded_text)

Original Tokens: ['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', '<|endoftext|>', 'UnknownWordExample']
Encoded IDs: [8, 142, 156, 59, 162, 12, 61, 576, 166, 839]
Decoded Text: 1 The Verdict Edith Wharton 1908 Exported from Wikisource on


In [12]:
# we have introduced 2 tokens here , unknow(unk) and endoftext
# unknow is used if any of the input is not in the vocab
# endoftext is used at the end of the sentence or new source of data

In [26]:
# special tokens special context tokens
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])
vocab={token:integer for integer,token in enumerate(all_tokens)}
len(vocab.items())

1278

In [25]:
#this for testing purpose where , if any of the strint is not match to vocab
#that we trained then it will mapp to ukknown

# Instantiate tokenizer
tokenizer = SimpleTokenizer(vocab=vocab)

# Encode the unknown word "pavan"
encoded_pavan = tokenizer.encode(["Verdict"])
unk_id = vocab['<|unk|>']

# Print evaluation results
print(f"Encoded ID for 'pavan': {encoded_pavan}")
print(f"ID of '<|unk|>': {unk_id}")
print(f"Is 'pavan' correctly mapped to '<|unk|>'? {encoded_pavan == [unk_id]}")

Encoded ID for 'pavan': [156]
ID of '<|unk|>': 1277
Is 'pavan' correctly mapped to '<|unk|>'? False


#Tokenization Algorithm

# **1) word based tokenization- every word is a token **

problems:

1.   what do you do if user enter word which are not in the vocab
2.   when we do word based token simliar words are not group together , they might get token way a far from each other (ex boy , boys both are almost similiar but id's will get so far away )

# **2) character based tokenization:**


1.   each character treats as token
never run of  out of voca(oov) since english has 256 characters


**problems: **


1.   the meaning associated with words are lost
2.   tokenization is larger than initial text

# 3) **sub word tokenization**
split based on rules


*   rule1: donot split frequenlt used words into smaller sub words
*   rule2: split the rare words into smaller meaningful subwords
   
adv:
1.    it handles outof vacob words very well





# **BPE( byte pair encoding ) is a sub word tokenization algorigthm**





In [15]:
# byte pair encodning
!pip3 install tiktoken

In [37]:
import tiktoken
import importlib

In [28]:
tokenizer =tiktoken.get_encoding("gpt2")

In [35]:
text=("hello , do you like tea?? <|endoftext|> in the sulit terraces")
Integers=tokenizer.encode(text,allowed_special={"<|endoftext|>"})

In [40]:
print(Integers)

[31373, 837, 466, 345, 588, 8887, 3548, 220, 50256, 287, 262, 33154, 270, 8812, 2114]


In [42]:
strings=tokenizer.decode(Integers)
print(strings)

hello , do you like tea?? <|endoftext|> in the sulit terraces


In [ ]:
#data sampling